# 10 — The mechanism claim as a registered test, on prompts nothing has touched

**Why this notebook exists.** Notebook 08 §2 asked whether prompt-gating and generation-gating are
equivalent routes to the same behaviour, and its rule tested **suppression** — which counts a
degenerate apology loop as a success, because the user did not get an answer either way. It fired
`REDUNDANT`. On **clean refusal** the same data separates the two conditions decisively, and reading
the text plus the detector's own feature margins showed why (WORKLOG 20).

That correction was made with the data in view, so it is recorded as a correction and **not** as a
verdict. This notebook turns it into a test: the rule is written below before anything runs, on the
axis every other damage-regime rule in this project uses, and it runs on the **48 prompts of
`test_n=144` that no run, mask, threshold or selection step has ever seen**.

Four arms at relative strength 2.0 — `all`, `prompt-only`, `gen-only`, and `prompt-last` as the
control that keeps "the prompt sets the effect" from meaning "any prompt position will do".
About ten minutes on a GPU.

**One dependency worth knowing.** The coherence detector's thresholds were fitted on layer-16
repetition loops, and gen-only produces a failure mode they under-fire on. If `notebooks/09` has
already produced re-fitted thresholds, §1.2 picks them up automatically and the verdict is computed
under them; if not, it runs on the original anchors and says so. Both rate sets are reported either
way, because the difference between them is exactly the instrument risk this comparison carries.

## §0 Setup — identical to notebooks 07 and 08

In [ ]:
# %% 0.0 BOOTSTRAP -- run this first, always. Identical locally and on Colab.
import os, subprocess, sys
from pathlib import Path

GITHUB_REPO = "YarinShitrit/adass"                 # from `git remote -v`
DRIVE_DIR   = "/content/drive/MyDrive/adass"       # fallback if you skip GitHub

IN_COLAB = "google.colab" in sys.modules


def _find_root(start):
    """Walk up looking for the repo: a pyproject.toml sitting next to the adass package."""
    p = Path(start).resolve()
    for cand in (p, *p.parents):
        if (cand / "pyproject.toml").is_file() and (cand / "adass" / "core.py").is_file():
            return cand
    return None


def _early_env():
    """Read a .env BEFORE the package exists. Mirrors adass.env, which is the canonical copy.

    Duplicated here because on Colab this cell runs before the repo is cloned and before anything
    is pip-installed -- and the GitHub token needed to perform the clone has to come from
    somewhere. Which is also why the repo's own .env cannot be that somewhere: .env is gitignored,
    so a clone never contains one. Keep a filled-in .env on Drive; it survives runtimes.
    """
    for p in (Path.cwd() / ".env", Path("/content/drive/MyDrive/adass/.env"),
              Path("/content/drive/MyDrive/.env"), Path("/content/.env"), Path.home() / ".env"):
        if p.is_file():
            for line in p.read_text(encoding="utf-8").splitlines():
                line = line.strip().removeprefix("export ")
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, _, v = line.partition("=")
                v = v.strip().strip("\"'")
                if v and not os.environ.get(k.strip()):
                    os.environ[k.strip()] = v
            print(f"loaded .env from {p}")
            return p
    return None


def _secret(name, prompt):
    """Environment (incl. .env) -> Colab Secrets -> prompt. Nothing is stored in the notebook."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata          # browser Colab frontend only
        v = userdata.get(name)
        if v:
            os.environ[name] = v
            return v
    except Exception:
        pass
    import getpass
    v = getpass.getpass(prompt)
    if v:
        os.environ[name] = v
    return v


def _clone_or_update(repo, token, dest):
    """Clone if absent, pull if already there. NEVER let the token reach a traceback.

    Two failures this exists for, both hit on 30 August. A kernel restart leaves /content intact,
    so `git clone` into an existing checkout dies with exit 128 and a message nobody sees; and
    `check=True` raises CalledProcessError, whose `.args` carries the tokenised URL straight into
    the traceback Colab prints and then saves into the notebook file. A leaked PAT is a worse
    outcome than a failed clone, so the token never travels with the exception.
    """
    dest = Path(dest)
    if (dest / ".git").is_dir():
        cmd, what = ["git", "-C", str(dest), "pull", "--quiet"], "pull"
    elif dest.exists() and any(dest.iterdir()):
        raise RuntimeError(f"{dest} exists and is not a git checkout. Remove it, or point ROOT at "
                           "the repo by hand.")
    else:
        cmd, what = ["git", "clone", "--quiet",
                     f"https://{token}@github.com/{repo}.git", str(dest)], "clone"
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        err = (r.stderr or "").strip()
        if token:
            err = err.replace(token, "<token>")
        raise RuntimeError(f"git {what} failed (exit {r.returncode}): {err[:400]}")
    print(f"git {what} ok -> {dest}")
    return dest


_early_env()
ROOT = _find_root(Path.cwd())

if IN_COLAB and ROOT is None:
    if not os.environ.get("GH_TOKEN") and Path("/content/drive").exists() is False:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            _early_env()
        except Exception:
            pass
    token = _secret("GH_TOKEN", "GitHub PAT (read access to the repo): ")
    if token:
        ROOT = _clone_or_update(GITHUB_REPO, token, "/content/adass")
    else:
        ROOT = _find_root(DRIVE_DIR) or Path(DRIVE_DIR)

assert ROOT is not None, "repo not found -- set GITHUB_REPO, or put the repo at DRIVE_DIR"
os.chdir(ROOT)

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
elif str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import adass
adass.load_env()
# Both sources are GATED: HF_TOKEN needs google/gemma-2-2b-it AND walledai/AdvBench accepted.
# NOTE: HF_HUB_OFFLINE is deliberately NOT set -- nothing is cached on a fresh runtime.
from huggingface_hub import get_token
if not get_token():
    adass.require("HF_TOKEN", "HuggingFace token (gemma-2-2b-it + AdvBench accepted): ")

import torch
print(adass.paths.describe())
print(adass.env.status())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none -- everything past §1 will be very slow")

In [ ]:
# %% 0.1 Run flags, the truncation point, and the prior run.
import json, math, itertools
from collections import Counter

OUT = "week7_registered.json"      # adass.save_results resolves bare names to data/results/

LOAD_MODEL = os.environ.get("ADASS_LOAD_MODEL", "0") == "1"
print(f"LOAD_MODEL={LOAD_MODEL}   (set ADASS_LOAD_MODEL=1 for everything past §1)")
RESULTS = {}

# save_results MERGES at the top level, so no later cell can delete a section it did not compute.
# §0.1 is the one deliberate truncation point, and it fires only on a full run: a CPU-only pass
# cannot regenerate the GPU sections, so rotating them away would destroy the only copy. That is
# not hypothetical -- it happened on 21 August, and again in a milder form on 25 August, which is
# why the `and LOAD_MODEL` is there.
_out = adass.results_path(OUT)
_prev = _out.with_suffix(".prev.json")
if _out.exists() and LOAD_MODEL:
    _out.replace(_prev)
    print(f"rotated {_out.name} -> {_prev.name}  (full run: starting clean)")
elif _out.exists():
    print(f"CPU-only run: MERGING into existing {_out.name}, not rotating.")

# The verdict cells re-run on CPU against whatever the last GPU run left behind, so every one of
# them reads through `stored()` rather than off a local variable that only exists mid-run.
# WHERE THE RESULTS SURVIVE. A Colab runtime takes its filesystem with it when it is recycled,
# and on 27 August it did exactly that to a completed run: every section had saved, every save had
# gone to /content/adass, and /content/adass no longer existed. Only the notebook's printed cell
# outputs were left -- the rates, but none of the generations or per-item instrument arrays.
#
# adass.save_results mirrors every write to ADASS_MIRROR when that is set. Point it at Drive and
# the copy outlives the runtime. This block sets it up automatically on Colab and says so loudly
# when it cannot, because a run that is not mirrored is a run you may have to pay for twice.
if IN_COLAB and not os.environ.get("ADASS_MIRROR"):
    _drive = Path("/content/drive/MyDrive")
    if not _drive.exists():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as _e:
            print(f"could not mount Drive ({_e})")
    if _drive.exists():
        os.environ["ADASS_MIRROR"] = str(_drive / "adass_results")
        print(f"mirroring every save to {os.environ['ADASS_MIRROR']}")
    else:
        print("!! NO MIRROR: results live only on this runtime and die with it.")
        print("!! Set ADASS_MIRROR to a Drive path, or copy data/results/ out before disconnecting.")
elif os.environ.get("ADASS_MIRROR"):
    print(f"mirroring every save to {os.environ['ADASS_MIRROR']}")

PRIOR = {}
for _p in (_out, _prev):
    if _p.exists():
        PRIOR = json.load(open(_p))
        print(f"prior run loaded from {_p.name}: {list(PRIOR)}")
        break


def stored(key, default=None):
    """This run's value if this run computed it, else the previous run's."""
    return RESULTS.get(key, PRIOR.get(key, default))

In [ ]:
# %% 0.2 Module provenance. The stored judge output is only reusable if the prompts still hash
# to the version it was produced under -- see HANDOVER trap 2 for what silent drift cost here.
print("adass      ", adass.__version__, "from", Path(adass.__file__).parent)
print("judge hash ", adass.judge_prompt_hash())
_stored_hash = json.load(open(adass.artifact("steps123_results.json")))["step3"]["prompt_hash"]
print("stored hash", _stored_hash,
      "-- MATCH" if adass.judge_prompt_hash() == _stored_hash else "-- CHANGED: do not reload")
assert adass.judge_prompt_hash() == _stored_hash, (
    "judge prompts changed: every comparison in this notebook against a stored week-4 number "
    "would be measuring two different instruments. Bump the version deliberately or revert.")

In [ ]:
# %% 0.3 Environment, splits, vectors. float16 is a STOP condition: Gemma-2 emits broken text in
# fp16, and that failure is visually identical to the degeneration this project studies.
import torch, transformers

DEV, DT = adass.pick_device(), adass.pick_dtype(adass.pick_device())
print("device", DEV, "| dtype", DT)
assert DT is not torch.float16, "float16: STOP. See README, environment check."

CONFIG = json.load(open(adass.paths.config()))
LAYER  = CONFIG["best_layer"]              # 10, set from evidence on 24 August
R16    = CONFIG["r16"]                     # 0.5537 -- relative strength of the raw vector at L16
assert LAYER == 10, f"config says layer {LAYER}; this notebook is written for the layer-10 point"

SPL     = adass.make_splits(seed=CONFIG["seed"])   # train_n MUST stay at its default 128 --
PROMPTS = SPL["harmless_test"]                     # train_n=160 shifts harmless_test by 32 items
assert len(PROMPTS) == 48
DIRS = torch.load(adass.artifact("refusal_dirs.pt"))
V    = DIRS[LAYER + 1]                             # refusal_dirs is indexed [layer + 1]
print(f"{len(PROMPTS)} test prompts | layer {LAYER} | ||V|| {float(V.norm()):.3f} "
      f"| r16 {R16:.4f}")

MAXNEW  = 128          # 48 tokens cannot show apology-then-answer; week 3 §2 is why this is 128
GEN_BS  = 8
KL_BS   = 4            # teacher-forced logits are [B, T, 256k]; 4 keeps a T4 inside its memory
KL_REF_TOKENS = 48     # the fixed reference text, as in week 3 §5.1
KL_WINDOW = 8          # see §5: the shared window that makes position schemes comparable

RESULTS["env"] = dict(load_model=LOAD_MODEL, device=DEV, dtype=str(DT), torch=torch.__version__,
                      transformers=transformers.__version__, layer=LAYER, r16=R16,
                      max_new_tokens=MAXNEW, n_prompts=len(PROMPTS),
                      gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
                      bf16_supported=(torch.cuda.is_bf16_supported()
                                      if torch.cuda.is_available() else None))
print("env:", RESULTS["env"])
print(adass.save_results(RESULTS, OUT))

MODEL = TOK = TO_CHAT = None
if LOAD_MODEL:
    MODEL, TOK, DT, DEV = adass.load_model()
    TO_CHAT = adass.make_chat_fn(TOK)
    print("model loaded")

## §1 Instruments, thresholds, and the gate

Same scorers and the same blocking gate as the two notebooks before this one. The only addition is
§1.2, which swaps in the re-fitted coherence thresholds when they exist.

In [ ]:
# %% 1.1 Fit the mechanical thresholds on the anchors, and define the two axes.
GENS = json.load(open(adass.artifact("week3_generations.json")))
FIT = adass.fit_coherence_thresholds(GENS["no-steer"], GENS["dense/all m=2"])
for feat, d in FIT.items():
    print(f"  {feat:12} thr={d['threshold']:8.3f}  bacc={d['balanced_acc']:.3f}  margin={d['margin']:+.3f}")


def mech_broken(texts):
    return [adass.classify_mechanical(t, FIT)["broken"] for t in texts]


def judge_answered(prompts, texts):
    """None when the model is not loaded, so every section still completes."""
    if not LOAD_MODEL:
        return None
    return [o["answered"] for o in
            adass.local_judge_binary(MODEL, TOK, TO_CHAT, list(zip(prompts, texts)))]


def score(texts, prompts, answered=None):
    """The row every table in this notebook reports. Wilson CIs on all three rates.

    CIs are stored as [lo, hi] -- wilson_ci returns (point, lo, hi) and the point estimate is
    already the neighbouring field. Indexing rather than unpacking that tuple is what cost a
    blocking control its meaning on 23 August (WORKLOG correction 14), so the slice is explicit.
    """
    br = mech_broken(texts)
    ans = judge_answered(prompts, texts) if answered is None else answered
    n = len(texts)
    row = dict(n=n, broken=sum(br) / n, broken_ci=list(adass.wilson_ci(sum(br), n))[1:],
               matcher=adass.refusal_rate(texts))
    if ans is not None:
        clean = [(not b) and (not a) for b, a in zip(br, ans)]
        row.update(suppressed=1 - sum(ans) / n,
                   suppressed_ci=list(adass.wilson_ci(n - sum(ans), n))[1:],
                   clean_refusal=sum(clean) / n,
                   clean_refusal_ci=list(adass.wilson_ci(sum(clean), n))[1:],
                   judge_answered=ans)
    row["mech_broken"] = br
    return row


def fmt(row, label=""):
    s = f"{label:30} broken {row['broken']:6.1%}"
    if "clean_refusal" in row:
        s += f" | suppressed {row['suppressed']:6.1%} | CLEAN {row['clean_refusal']:6.1%}"
    if "kl" in row:
        s += f" | KL {row['kl']:6.3f}"
    return s + f" | matcher {row['matcher']:6.1%}"


def disjoint(a, b):
    """Do two [lo, hi] intervals fail to overlap? The only evidence a cell is DECIDED at n=48."""
    return a[1] < b[0] or b[1] < a[0]

In [ ]:
# %% 1.2 Use the re-fitted thresholds if notebook 09 has produced them.
#
# The detector was fitted on `no-steer` against `dense/all m=2`, both layer 16, and gen-only's
# apology loops clear all three thresholds while sitting just under each (WORKLOG 20). A verdict
# about gen-only computed on those thresholds inherits that slack, so if better ones exist, use
# them -- and report both rates regardless, because the gap between them IS the instrument risk.
FIT_ANCHOR = dict(FIT)
REFIT_SOURCE = "week3 anchors (no-steer vs dense/all m=2)"
try:
    _lab = json.load(open(adass.artifact("week6_labels.json")))
    _new = _lab.get("new_thresholds")
    if _new:
        FIT = {k: dict(FIT_ANCHOR[k], threshold=v) for k, v in _new.items()}
        REFIT_SOURCE = "week6 hand labels (notebook 09)"
except FileNotFoundError:
    pass
print(f"coherence thresholds in use: {REFIT_SOURCE}")
for k in FIT:
    print(f"   {k:12s} {FIT_ANCHOR[k]['threshold']:8.3f} -> {FIT[k]['threshold']:8.3f}")


def mech_broken_with(texts, fit):
    return [adass.classify_mechanical(t, fit)["broken"] for t in texts]

In [ ]:
# %% 1.2 The replication gate, plus the negative control and the KL reference text.
#
# Three things at once, all from the unsteered and the dense rel-1.0 conditions:
#   - REF_TEXTS  -- the fixed unsteered continuation every KL in this notebook is measured on;
#   - the negative control -- unsteered must be ~0% suppressed and ~0% broken, or the
#     instruments are wrong before any comparison starts;
#   - the gate -- dense at rel 1.0 must land where week 4 §7 left it.
if LOAD_MODEL:
    HN10 = adass.mean_hidden_norm(MODEL, TOK, TO_CHAT, PROMPTS, LAYER, device=DEV)
    print(f"mean ||h|| at layer {LAYER}: {HN10:.1f}  (week 4 measured 170.9)")

    REF_TEXTS = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, mult=0.0,
                               max_new_tokens=KL_REF_TOKENS, batch_size=GEN_BS,
                               device=DEV, dtype=DT)
    ns_gens = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, mult=0.0, max_new_tokens=MAXNEW,
                             batch_size=GEN_BS, device=DEV, dtype=DT)
    ns = score(ns_gens, PROMPTS)
    print(fmt(ns, "no-steer (negative control)"))

    V_ref = adass.rel_norm_rows(V, R16 * 1.0, HN10)      # the operating point, norm-matched
    d10_gens = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, layer=LAYER, vector=V_ref, mult=1.0,
                              positions="all", max_new_tokens=MAXNEW, batch_size=GEN_BS,
                              device=DEV, dtype=DT)
    d10 = score(d10_gens, PROMPTS)
    d10.update(adass.strength_row(V_ref, 1.0, HN10), label="dense", rel_factor=1.0,
               method="dense", sparsity=0.0, positions="all", n_prompts=len(PROMPTS))
    print(fmt(d10, "dense/all rel 1.0"))

    w4 = json.load(open(adass.artifact("week4_layers.json")))["s7_relative_grid"]["cells"]["L10/rel1.0"]["row"]
    print(f"\nweek 4 §7 L10/rel1.0: broken {w4['broken']:.1%}  clean {w4['clean_refusal']:.1%}")

    # The bar is 10 points, not equality: the 24 August run reproduced every rate in the sweep to
    # within 4.2% while only 14-48% of individual generations matched token-for-token, because a
    # T4 has no bfloat16 and greedy decoding is deterministic given identical numerics and not
    # otherwise. Rates are dtype-invariant here; text is not.
    gate = dict(
        neg_control_ok=bool(ns["broken"] <= 0.05 and ns["suppressed"] <= 0.05),
        clean_delta=d10["clean_refusal"] - w4["clean_refusal"],
        broken_delta=d10["broken"] - w4["broken"],
        h_norm=HN10, h_norm_w4=CONFIG["h_norms"][str(LAYER)])
    gate["repro_ok"] = bool(abs(gate["clean_delta"]) <= 0.10 and abs(gate["broken_delta"]) <= 0.10)
    gate["pass_"] = bool(gate["neg_control_ok"] and gate["repro_ok"])
    print(f"\nnegative control {'PASS' if gate['neg_control_ok'] else 'FAIL'} | "
          f"replication delta clean {gate['clean_delta']:+.1%} broken {gate['broken_delta']:+.1%} "
          f"-> {'PASS' if gate['repro_ok'] else 'FAIL'}")
    RESULTS["s1_gate"] = dict(gate=gate, no_steer=ns, dense_rel1=d10)
    RESULTS["s1_ref_texts"] = REF_TEXTS
    print(adass.save_results(RESULTS, OUT))
    assert gate["pass_"], "BLOCKING: fix this before running anything below."
else:
    HN10 = CONFIG["h_norms"][str(LAYER)]
    REF_TEXTS = (PRIOR.get("s1_ref_texts") or None)
    ns = d10 = None
    print(f"deferred: needs ADASS_LOAD_MODEL=1. Using stored ||h|| = {HN10}")

In [ ]:
# %% 1.3 The fresh prompts, and the runner.
if LOAD_MODEL:
    SPL144 = adass.make_splits(seed=CONFIG["seed"], test_n=144)
    P144 = SPL144["harmless_test"]
    assert P144[:48] == PROMPTS, "the extended split moved the original 48 -- STOP"
    assert P144[:96] == adass.make_splits(seed=CONFIG["seed"], test_n=96)["harmless_test"], \
        "the extended split moved the week-5 96 -- STOP"
    FRESH = P144[96:]
    # This run has the token and the datasets in hand, so it is the right place to check the cached
    # prompt list the CPU-only notebooks read instead. If these ever disagree, notebook 09's labels
    # are keyed to the wrong prompts and its sheet is void.
    _cached = json.load(open(adass.artifact("harmless_test_prompts.json")))["prompts"]
    assert _cached == PROMPTS, "harmless_test_prompts.json has drifted from make_splits -- STOP"
    print("cached prompt list matches make_splits")
    print(f"{len(FRESH)} prompts, unseen by every run, mask, threshold and selection step so far")
    print("first two:", *[f"  - {p}" for p in FRESH[:2]], sep="\n")

REL = json.load(open(adass.artifact("week5_h1h3.json")))["s2_damage_onset"]["rel_star"]
print(f"relative strength for every arm below: x{REL} (REL_STAR, from week 5)")


def run_arm(label, positions, prompts=None):
    prompts = FRESH if prompts is None else prompts
    vec = adass.rel_norm_rows(V, R16 * REL, HN10)
    g = adass.generate(MODEL, TOK, TO_CHAT, prompts, layer=LAYER, vector=vec, mult=1.0,
                       positions=positions, max_new_tokens=MAXNEW, batch_size=GEN_BS,
                       device=DEV, dtype=DT)
    row = score(g, prompts)                       # uses FIT, i.e. the re-fitted thresholds
    row.update(adass.strength_row(vec, 1.0, HN10), label=label, positions=str(positions),
               rel_factor=REL, n_prompts=len(prompts))
    # the same rates under the ORIGINAL thresholds, so the instrument risk is visible per row
    ob = mech_broken_with(g, FIT_ANCHOR)
    row["broken_anchor"] = sum(ob) / len(ob)
    if "judge_answered" in row:
        row["clean_refusal_anchor"] = sum(
            (not b) and (not a) for b, a in zip(ob, row["judge_answered"])) / len(ob)
    adass.empty_cache(DEV)
    return row, g

## §2 The registered rule

Written before the run, on the axis notebook 08 §2 should have used.

**Primary axis: clean refusal** — coherent *and* not answered. Suppression alone cannot distinguish
refusing from breaking, which is the failure this entire project documents and the reason the earlier
rule returned an answer its own data did not support.

> **DISSOCIATION CONFIRMED** if both hold:
> - **(a) the routes differ in quality of effect** — `prompt-only`'s clean-refusal interval is
>   disjoint from `gen-only`'s, with prompt-only higher;
> - **(b) the damage is specific to generation-time steering** — `prompt-only` breaks ≤ 10%, and its
>   broken interval is disjoint from `gen-only`'s.
>
> **EFFECT ONLY** if (a) holds and (b) does not — the two routes differ, but the damage does not
> localise to generated positions and the mechanism must be stated without that half.
>
> **DAMAGE ONLY** if (b) holds and (a) does not.
>
> **NOT CONFIRMED** if neither holds. At n=48 per arm this would be reported as what it is: on the
> gap the 30 August run measured (97.9% against 68.8%) these intervals separate comfortably, so a
> failure here is a real disagreement between prompt sets, not a power problem.

**Suppression is recorded but decides nothing.** It is reported for every arm precisely so that the
redundancy the earlier rule found can be seen to reproduce — both routes suppress answering, and only
one of them does it without wrecking the text. That is the whole point, and stating both halves is
what makes it a finding rather than a correction.

**The control.** `prompt-last` steers a single prompt position. If "the effect is set at the prompt"
degenerates into "any prompt position will do", this arm would suppress as much as `prompt-only`; on
the original 48 it suppressed 27.1%, so it does not.

In [ ]:
# %% 2.1 The four arms, on the fresh prompts.
ARMS = [("all", "all"), ("prompt-only", "prompt_only"),
        ("gen-only", "gen_only"), ("prompt-last", "prompt_last")]

if LOAD_MODEL:
    arms = {}
    for label, spec in ARMS:
        row, gens = run_arm(label, spec)
        arms[label] = dict(row=row, gens=gens)
        print(f"{fmt(row, f'{label:12s} rel x{REL} (fresh n=48)')}"
              f" | broken@anchor {row['broken_anchor']:5.1%}")
    RESULTS["s2_arms"] = dict(cells=arms, rel_factor=REL, n=len(FRESH),
                              prompts_from="harmless_test[96:144]",
                              thresholds=REFIT_SOURCE)
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred: needs ADASS_LOAD_MODEL=1")

In [ ]:
# %% 2.2 The registered verdict.
s2 = stored("s2_arms")
if s2:
    R = {k: v["row"] for k, v in s2["cells"].items()}
    print(f"{'arm':13s} {'clean refusal':>14s} {'interval':>14s} {'broken':>8s} {'suppressed':>11s}")
    for k, r in R.items():
        ci = f"[{r['clean_refusal_ci'][0]:.2f},{r['clean_refusal_ci'][1]:.2f}]"
        print(f"{k:13s} {r['clean_refusal']:14.1%} {ci:>14s} {r['broken']:8.1%} {r['suppressed']:11.1%}")

    po, go = R["prompt-only"], R["gen-only"]
    a = disjoint(po["clean_refusal_ci"], go["clean_refusal_ci"]) and \
        po["clean_refusal"] > go["clean_refusal"]
    b = po["broken"] <= 0.10 and disjoint(po["broken_ci"], go["broken_ci"])
    verdict = ("DISSOCIATION CONFIRMED" if (a and b) else
               "EFFECT ONLY -- the routes differ; the damage does not localise" if a else
               "DAMAGE ONLY -- damage localises; the routes do not differ in quality" if b else
               "NOT CONFIRMED on unseen prompts")
    print(f"\n(a) clean refusal, prompt-only > gen-only, disjoint : {a}")
    print(f"(b) prompt-only broken <= 10% and disjoint from gen-only: {b}")
    print(f"\nVERDICT: {verdict}")

    # Recorded, deciding nothing: does the redundancy the earlier rule found reproduce?
    redundant_on_suppression = not disjoint(po["suppressed_ci"], go["suppressed_ci"])
    print(f"\nBoth routes suppress answering (the earlier rule's finding): "
          f"{redundant_on_suppression} -- prompt-only {po['suppressed']:.1%}, "
          f"gen-only {go['suppressed']:.1%}")
    print("Reported because it is half the result: the routes are equivalent in whether the user "
          "gets\nan answer, and not equivalent in what they leave behind.")

    # And the same verdict computed on the ORIGINAL thresholds, so the instrument risk is explicit.
    if "clean_refusal_anchor" in po:
        n = po["n"]
        pa, ga = po["clean_refusal_anchor"], go["clean_refusal_anchor"]
        ca = list(adass.wilson_ci(round(pa * n), n))[1:]
        cg = list(adass.wilson_ci(round(ga * n), n))[1:]
        print(f"\nunder the ORIGINAL thresholds: prompt-only {pa:.1%} vs gen-only {ga:.1%}, "
              f"disjoint={disjoint(ca, cg)}")

    RESULTS["s2_verdict"] = dict(verdict=verdict, effect_differs=bool(a), damage_localises=bool(b),
                                 redundant_on_suppression=bool(redundant_on_suppression),
                                 thresholds=s2["thresholds"], n=s2["n"],
                                 axis="clean refusal (registered in §2 above, before the run)")
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred: §2.1 has not run")

## §3 Against the 30 August run

Same four arms, same relative strength, different 48 prompts. This is the second thing the notebook
buys: the 30 August numbers were measured on the prompt set every earlier run used, and a claim that
holds on one set of 48 alpaca instructions and not another is a claim about the prompts.

Differences of a few points are expected — these are independent samples of 48, not a re-run of the
same one. What would matter is a **sign change** or a collapse in the gap.

In [ ]:
# %% 3.1 Side by side.
s2 = stored("s2_arms")
if s2:
    W5 = json.load(open(adass.artifact("week5_h1h3.json")))
    W6 = json.load(open(adass.artifact("week6_mechanism.json")))
    OLD = {"all": W5["s5_positions"]["cells"][f"all/rel{REL}"]["row"],
           "prompt-only": W5["s5_positions"]["cells"][f"prompt-only/rel{REL}"]["row"],
           "prompt-last": W5["s5_positions"]["cells"][f"prompt-last/rel{REL}"]["row"],
           "gen-only": W6["s2_gen_only"]["cells"][f"gen-only/rel{REL}"]["row"]}
    NEW = {k: v["row"] for k, v in s2["cells"].items()}
    print(f"{'arm':13s} {'clean 30 Aug':>13s} {'clean fresh':>12s} {'delta':>8s}"
          f" {'broken 30 Aug':>14s} {'broken fresh':>13s}")
    rows = []
    for k in NEW:
        o, n = OLD[k], NEW[k]
        d = n["clean_refusal"] - o["clean_refusal"]
        rows.append(dict(arm=k, clean_old=o["clean_refusal"], clean_new=n["clean_refusal"],
                         delta=d, broken_old=o["broken"], broken_new=n["broken"]))
        print(f"{k:13s} {o['clean_refusal']:13.1%} {n['clean_refusal']:12.1%} {d:+8.1%}"
              f" {o['broken']:14.1%} {n['broken']:13.1%}")
    gap_old = OLD["prompt-only"]["clean_refusal"] - OLD["gen-only"]["clean_refusal"]
    gap_new = NEW["prompt-only"]["clean_refusal"] - NEW["gen-only"]["clean_refusal"]
    print(f"\nprompt-only minus gen-only, clean refusal: {gap_old:+.1%} on 30 August, "
          f"{gap_new:+.1%} here")
    RESULTS["s3_vs_prior"] = dict(rows=rows, gap_old=gap_old, gap_new=gap_new)
    print(adass.save_results(RESULTS, OUT))

    print("\nRead four (HANDOVER trap 8):")
    for k in ("prompt-only", "gen-only"):
        for i in (0, 1):
            print(f"\n--- {k}, prompt: {FRESH[i] if LOAD_MODEL else '(model not loaded)'}")
            print(s2["cells"][k]["gens"][i][:300])
else:
    print("deferred: §2.1 has not run")

## §4 What this notebook settled

In [ ]:
# %% 4.1 Summary.
print("=" * 74)
for key, name in [("s1_gate", "replication gate"), ("s2_verdict", "registered mechanism test"),
                  ("s3_vs_prior", "against 30 August")]:
    s = stored(key)
    if not s:
        print(f"{name:28s} not run")
    elif "verdict" in s:
        print(f"{name:28s} {s['verdict']}")
    elif key == "s1_gate":
        print(f"{name:28s} {'PASS' if s['gate']['pass_'] else 'FAIL'}")
    else:
        print(f"{name:28s} gap {s['gap_old']:+.1%} -> {s['gap_new']:+.1%}")
print("=" * 74)
RESULTS["meta"] = dict(notebook="10_registered_mechanism", layer=LAYER, rel_factor=REL,
                       prompts="harmless_test[96:144], unseen by every prior run",
                       registered="§2's rule was written above its code before the run, on clean "
                                  "refusal -- the axis notebook 08 §2 should have used")
print(adass.save_results(RESULTS, OUT))